Install Tapis Python SDK.  After running the code below you need to restart the runtime - go to the Menu and select Runtime -> Restart runtime or use CTRL+M on the keyboard. Now you can execute the code in the notebook and follow the rest of the tutorial.

In [ ]:
# !pip install -q tapipy

In [ ]:
from app_parameters_config import (
    AMPLISEQ_TEST_ARGS,
    AMPLISEQ_CONDENSED_AMP_ARGS,
    AMPLISEQ_ITS_AMP_ARGS,
    AMPLISEQ_16S_AMP_ARGS 
)

## Enter TAPIS user and host machine information

To get things started, please run the following and enter the training account information provided to you:

In [ ]:
# For use on CI-PP
portal_tenant = 'cipp'

username = "andyyu"
password = "P#nguin777"

# UH Tenant
base_url = 'https://uhprod.uhtapis.org'

host_username = 'cmaiki_service'
host = 'koa.its.hawaii.edu'

## Authenticate and initialize Tapis v3 client

Using this information, you can now use `tapipy` to authenticate in the tenant and initialize the
Tapis v3 client. You should see your token information displayed. This may take a while to run but should take
no more than 30 seconds.

In [ ]:
from tapipy.tapis import Tapis
#Create python Tapis client for user
client = Tapis(base_url= base_url, username=username, password=password)
# *** Tapis v3: Call to Tokens API
client.get_tokens()
# Print Tapis v3 token
client.access_token

In [ ]:
# Export access_token to JWT env variable

import os
os.environ['JWT'] = client.access_token.access_token

In [ ]:
# client.tenants.list_tenants()

## Systems

### Create a system for the HPC cluster

With just a few changes to the system definition you can create a system that can be used to run the
same application on an HPC type host. Note the minimal changes:

* **id** - A unique id is required
* **host** - Main hostname for the HPC system.
* **rootDir** - Using the root directory of the host gives us flexibility in setting **jobWorkingDir**.
  Note that you still need LINUX permissions.
* **jobWorkingDir** - Now determined dynamically using the Tapis v3 function HOST_EVAL()
* **jobRuntimes** - Most HPC systems support singularity and not docker
* **batchLogicalQueue.hpcQueueName** - HPC queue to use by default.
* **batchLogicalQueues** - HPC queue definitions for this HPC system.

### Schduler Profile
Typically not necessary

In [ ]:
user_id = username
root_dir = "/mnt/lustre/koa/koastore/cmaiki_group"

# CI-PP system
system_id_hpc = "cmaiki-on-CI-PP-koa-hpc"


# Create the system definition
exec_system_hpc = {
  "id": system_id_hpc,
  "description": "System for running C-MAIKI pipelines on Koa HPC cluster",
  "systemType": "LINUX",
  "host": host,
  "port": 2022,
  "defaultAuthnMethod": "PKI_KEYS",
  "effectiveUserId": host_username,
#   "effectiveUserId": "${apiUserId}",
  "rootDir": root_dir,
  "canExec": True,
  "jobRuntimes": [ { "runtimeType": "ZIP" } ],
#   "jobWorkingDir": "${EffectiveUserId}",
  "jobWorkingDir": "cmaiki_service",
    "canRunBatch": True,
  "batchScheduler": "SLURM",
  "batchDefaultLogicalQueue": "shared",
  "batchLogicalQueues": [
    {
      "name": "koa-shared",
      "hpcQueueName": "shared",
      "maxJobs": 50,
      "maxJobsPerUser": 10,
      "minNodeCount": 1,
      "maxNodeCount": 1,
      "minCoresPerNode": 1,
      "maxCoresPerNode": 1,
      "minMemoryMB": 3000,
      "maxMemoryMB": 32000,
      "minMinutes": 1,
      "maxMinutes": 4320
    }
  ]
}

# Use the client to create the system in Tapis
# print("****************************************************")
# print("Create system: " + system_id_hpc)
# print("****************************************************")
# client.systems.createSystem(**exec_system_hpc)

# If you need to update the system, modify the above definition as needed
client.systems.patchSystem(**exec_system_hpc, systemId=system_id_hpc)

In [ ]:
# # List all systems available to you
# print("****************************************************")
# print("List all systems")
# print("****************************************************")
# client.systems.getSystems()

In [ ]:
# ### Get details for the system you created
# print("****************************************************")
# print("Fetch system: " + system_id_hpc)
# print("****************************************************")
# client.systems.getSystem(systemId=system_id_hpc)

### Register Credentials for the HPC system

As before, now you will need to register credentials for your username. These will be used by Tapis to
access the host.

In [ ]:
private_key = os.getenv('CMAIKI_SERVICE_TAPIS_PRIVATE_KEY')
public_key = os.getenv('CMAIKI_SERVICE_TAPIS_PUBLIC_KEY')

In [ ]:
# Register credentials
client.systems.createUserCredential(systemId=system_id_hpc, userName=host_username, publicKey=public_key, privateKey=private_key)

## Application

In order to run a job on a system you will need to create a Tapis application.

### Create an application that can be run on the VM host or the HPC cluster

In [ ]:
# app_id_hpc = "cmaiki_service_cleanup"

# app_def_hpc= {
#     "id": app_id_hpc,
#     "version": "0.1",
#     "description": "Clean up files generated by cmaiki_service that lack group permissions.",
#     "runtime": "ZIP",
#     "runtimeOptions": ["NONE"],
#     "strictFileInputs": False,
#     "containerImage": "/mnt/lustre/koa/scratch/andyyu/apps/cmaiki_service_cleanup.tar.gz",
#     "jobType": "BATCH",
#     "jobAttributes": {
#         "archiveOnAppError": True,
#         "archiveSystemId": system_id_hpc,
#         "archiveSystemDir": "${JobOwner}/jobs/${JobUUID}",
#         "execSystemId": system_id_hpc,
#         "execSystemOutputDir": "${JobOwner}/jobs/${JobUUID}",
#         "parameterSet": {
#             "archiveFilter": {
#                 "includeLaunchFiles": True
#             },
#             "appArgs": [
#                 {"name": "max_mismatches", "arg": "--max_mismatches 3", "description": f"Number of allowed mismatched with (combined-paired-end) barcode(s) sequence", "inputMode": "REQUIRED"},
#             ]
#         },
#         "memoryMB": 16,
#         "nodeCount": 1,
#         "coresPerNode": 1,
#         "maxMinutes": 30
#     },
# }

## Deprecating
### 16S v1.0 App Def

In [ ]:
# app_id_hpc = "16Sv1-pipeline-uhhpc"

# app_def_hpc= {
#     "id": app_id_hpc,
#     "version": "1.0",
#     "description": "Analysis pipeline for bacterial 16S data.",
#     "runtime": "ZIP",
#     "runtimeOptions": ["NONE"],
#     "strictFileInputs": False,
#     "containerImage": "cmaiki_group/apps/16S-pipeline-app-v2.0.tar.gz",
#     "jobType": "BATCH",
#     "jobAttributes": {
#         "archiveOnAppError": True,
#         "archiveSystemId": system_id_hpc,
#         "archiveSystemDir": "${JobWorkingDir}/jobs/${JobUUID}",
#         "execSystemId": system_id_hpc,
#         "execSystemOutputDir": "${JobWorkingDir}/jobs/${JobUUID}",
#         "parameterSet": {
#             "archiveFilter": {
#                 "includeLaunchFiles": True
#             },
#             "appArgs":[
#                 {"name": "single_end", "arg": "--single_end", "description": "Process as single-end reads?", "inputMode": "INCLUDE_ON_DEMAND"},
#                 {"name": "trunc_fwd", "arg": "--trunc_fwd 220", "description": "Forward read truncation length in base pairs", "inputMode": "REQUIRED"},
#                 {"name": "trunc_rev", "arg": "--trunc_rev 190", "description": "Reverse read truncation length in base pairs", "inputMode": "REQUIRED"},
#                 {"name": "min_read_len", "arg": "--min_read_len 20", "description": "Minimum read length for filtering", "inputMode": "REQUIRED"},
#                 {"name": "max_expected_error", "arg": "--max_expected_error 3", "description": "Maximum number of expected errors per read (> 0)", "inputMode": "REQUIRED"},
#                 {"name": "pool", "arg": "--pool", "description": f"Pool samples for denoising (recommended for < 500 samples)", "inputMode": "INCLUDE_BY_DEFAULT", "notes": {"include":"True"}},
#                 {"name": "min_overlap", "arg": "--min_overlap 20", "description": "Minimum overlap for merging reads into contigs", "inputMode": "REQUIRED"},
#                 {"name": "max_mismatch", "arg": "--max_mismatch 1", "description": "Number of allowed mismatches between forward and reverse read when merging", "inputMode": "REQUIRED"},
#                 {"name": "min_abundance", "arg": "--min_abundance 2", "description": "Minimum OTU abundance", "inputMode": "REQUIRED"},
#                 {"name": "clustering_thresholds", "arg": "--clustering_thresholds 100,97", "description": "Sequence similarity thresholds for OTU clustering (comma separated)", "inputMode": "REQUIRED"},
#                 {"name": "skip_subsampling", "arg": "--skip_subsampling", "description": "Skip subsampling?", "inputMode": "INCLUDE_ON_DEMAND"},
#                 {"name": "custom_subsampling_level", "arg": "--custom_subsampling_level ", "description": "Exact subsampling level (overrides the next 2 parameters if set)", "inputMode": "REQUIRED"},
#                 {"name": "min_subsampling", "arg": "--min_subsampling 5000", "description": "Minimum number of sequences for subsampling. Used if the automatic subsampling level falls below this value", "inputMode": "REQUIRED"},
#                 {"name": "subsampling_quantile", "arg": "--subsampling_quantile 0.1", "description": "Automatic subsampling threshold (quantile of the sample sizes distributions). Ignored if an exact subsampling level is chosen", "inputMode": "REQUIRED"},
#                 {"name": "remove_unknown", "arg": "--remove_unknown", "description": "Remove unknown OTUs (at the domain level)?", "inputMode": "INCLUDE_BY_DEFAULT"},
#                 {"name": "remove_chloroplasts", "arg": "--remove_chloroplasts", "description": "Remove chloroplasts?", "inputMode": "INCLUDE_BY_DEFAULT"},
#                 {"name": "remove_mitochondria", "arg": "--remove_mitochondria", "description": "Remove mitochondria?", "inputMode": "INCLUDE_BY_DEFAULT"},
#                 {"name": "taxa_to_filter", "arg": "--taxa_to_filter", "description": "Experimental: Other taxa to filter? (Mothur syntax with commas instead of semicolons)", "inputMode": "REQUIRED"},
#                 {"name": "skip_unifrac", "arg": "--skip_unifrac", "description": "Skip tree and unifrac distances computation ? (significantly reduces computational time)", "inputMode": "INCLUDE_ON_DEMAND"},
#                 {"name": "alpha_diversity", "arg": "--alpha_diversity nseqs-sobs-chao-shannon-shannoneven", "description": "Alpha diversity metrics to compute (available in mothur)", "inputMode": "REQUIRED"},
#                 {"name": "beta_diversity", "arg": "--beta_diversity braycurtis-thetayc-sharedsobs-sharedchao", "description": "Beta diversity metrics to compute (available in mothur)", "inputMode": "REQUIRED"}
#             ]
#         },
#         "memoryMB": 32000,
#         "nodeCount": 1,
#         "coresPerNode": 1,
#         "maxMinutes": 120
#     },
# }

In [ ]:
# client.apps.createAppVersion(**app_def_hpc)

# client.apps.patchApp(**app_def_hpc, appId=app_id_hpc, appVersion='1.0')

In [ ]:
# # Submit job to run the application
# job_def_hpc = {
#     "name": "16S-pipeline-test-files",
#     "description":"Running the 16S pipeline with test files",
#     "appId": app_id_hpc,
#     "appVersion": '0.5',
#     "execSystemId":system_id_hpc,
#     "jobType": "BATCH",
# }

# # Submit a job
# job_response_hpc=client.jobs.submitJob(**job_def_hpc)

### 16S v0.0.2 App Def

In [ ]:
# app_id_hpc = "16S-pipeline-uhhpc"
app_id_hpc = "16S-v0.0.2-pipeline-uhhpc"
output_dir = "${JobOwner}/jobs/${JobUUID}"

app_def_hpc= {
    "id": app_id_hpc,
    "version": "0.0.2",
    "description": "Analysis pipeline for bacterial 16S data.",
    "runtime": "ZIP",
    "runtimeOptions": ["NONE"],
    "strictFileInputs": False,
    "containerImage": "/mnt/lustre/koa/lab/cmaiki_group/cmaiki_v2_apps/16S-pipeline-app-v0.0.2.tar.gz",
    "jobType": "BATCH",
    "jobAttributes": {
        "archiveOnAppError": True,
        "archiveSystemId": system_id_hpc,
        "archiveSystemDir": output_dir,
        "execSystemId": system_id_hpc,
        "execSystemOutputDir": output_dir,
        "parameterSet": {
            "archiveFilter": {
                "includeLaunchFiles": True
            },
            "appArgs":[
                {"name": "outdir", "arg": f"--outdir 16S-pipeline_outputs", "description": f"Output path", "inputMode": "REQUIRED", "notes": {"Hidden": "true"}},
                {"name": "db", "arg": "--db nr", "description": "Silva database type: \"nr\" or \"seed\"", "inputMode": "REQUIRED", "notes": {"Optional": "", "Dropdown": ["--db nr", "--db seed"]} },
#                 {"name": "minReads", "arg": "--minReads 50", "description": "Sample with less than <minRead> are discarded", "inputMode": "REQUIRED", "notes": {"Optional": ""} },
                {"name": "singleEnd", "arg": "--singleEnd", "description": "Process as single-end reads?", "inputMode": "INCLUDE_ON_DEMAND", "notes": {"Optional": ""} },
                {"name": "truncFwd", "arg": "--truncFwd 250", "description": "Forward read truncation length in base pairs", "inputMode": "REQUIRED", "notes": {"Optional": ""} },
                {"name": "truncRev", "arg": "--truncRev 190", "description": "Reverse read truncation length in base pairs", "inputMode": "REQUIRED", "notes": {"Optional": ""} },
                {"name": "minLength", "arg": "--minLength 20", "description": "Minimum read length for filtering", "inputMode": "REQUIRED", "notes": {"Optional": ""} },
                {"name": "maxEE", "arg": "--maxEE 3", "description": "Maximum number of expected errors per read (> 0)", "inputMode": "REQUIRED", "notes": {"Optional": ""} },
#                 {"name": "truncQ", "arg": "--truncQ 2", "description": "Read truncation at the 1st occurence of a base of quality <= <truncQ>", "inputMode": "REQUIRED", "notes": {"Optional": ""} },
                {"name": "minOverlap", "arg": "--minOverlap 20", "description": "Minimum overlap for merging reads into contigs", "inputMode": "REQUIRED", "notes": {"Optional": ""} },
                {"name": "maxMismatch", "arg": "--maxMismatch 1", "description": "Number of allowed mismatches between forward and reverse read when merging", "inputMode": "REQUIRED", "notes": {"Optional": ""} },
                {"name": "minAbundance", "arg": "--minAbundance 2", "description": "Minimum OTU abundance", "inputMode": "REQUIRED", "notes": {"Optional": ""} },
                {"name": "clusteringThresholds", "arg": "--clusteringThresholds 100,97", "description": "Sequence similarity thresholds for OTU clustering (comma separated)", "inputMode": "REQUIRED", "notes": {"Optional": ""} },
                {"name": "skipSubsampling", "arg": "--skipSubsampling", "description": "Skip subsampling?", "inputMode": "INCLUDE_ON_DEMAND", "notes": {"Optional": ""} },
                {"name": "customSubsamplingLevel", "arg": "--customSubsamplingLevel ", "description": "Exact subsampling level (overrides the next 2 parameters if set)", "inputMode": "REQUIRED", "notes": {"Optional": "true"} },
                {"name": "minSubsampling", "arg": "--minSubsampling 5000", "description": "Minimum number of sequences for subsampling. Used if the automatic subsampling level falls below this value", "inputMode": "REQUIRED", "notes": {"Optional": ""} },
                {"name": "subsamplingQuantile", "arg": "--subsamplingQuantile 0.1", "description": "Automatic subsampling threshold (quantile of the sample sizes distributions). Ignored if an exact subsampling level is chosen", "inputMode": "REQUIRED", "notes": {"Optional": ""} },
                {"name": "removeUnknown", "arg": "--removeUnknown", "description": "Remove unknown OTUs (at the domain level)?", "inputMode": "INCLUDE_BY_DEFAULT", "notes": {"Optional": ""} },
                {"name": "removeChloroplasts", "arg": "--removeChloroplasts", "description": "Remove chloroplasts?", "inputMode": "INCLUDE_BY_DEFAULT", "notes": {"Optional": ""} },
                {"name": "removeMitochondria", "arg": "--removeMitochondria", "description": "Remove mitochondria?", "inputMode": "INCLUDE_BY_DEFAULT", "notes": {"Optional": ""} },
                {"name": "taxaToFilter", "arg": "--taxaToFilter", "description": "Experimental: Other taxa to filter? (Mothur syntax with commas instead of semicolons)", "inputMode": "REQUIRED", "notes": {"Optional": ""} },
            ]
        },
        "memoryMB": 32000,
        "nodeCount": 1,
        "coresPerNode": 1,
        "maxMinutes": 240
    },
}

In [ ]:
# client.apps.createAppVersion(**app_def_hpc)

client.apps.patchApp(**app_def_hpc, appId=app_id_hpc, appVersion='0.0.2')

### ITS App Def

In [ ]:
app_id_hpc = "ITS-pipeline-uhhpc"
output_dir = "${JobOwner}/jobs/${JobUUID}"


app_def_hpc= {
    "id": app_id_hpc,
    "version": "1.0",
    "description": "ITS pipeline using test files.",
    "runtime": "ZIP",
    "runtimeOptions": ["NONE"],
    "strictFileInputs": False,
    "containerImage": "/mnt/lustre/koa/lab/cmaiki_group/cmaiki_v2_apps/ITS-pipeline-app-v2.0.tar.gz",
    "jobType": "BATCH",
    "jobAttributes": {
        "archiveOnAppError": True,
        "archiveSystemId": system_id_hpc,
        "archiveSystemDir": output_dir,
        "execSystemId": system_id_hpc,
        "execSystemOutputDir": output_dir,
        "parameterSet": {
            "archiveFilter": {
                "includeLaunchFiles": True
            },
            "appArgs": [
                {"name": "outdir", "arg": f"--outdir ITS-pipeline_outputs", "description": f"Output path", "inputMode": "REQUIRED", "notes": {"Hidden": "true"}},
                {"name": "locus", "arg": "--locus ITS1", "description": f"Which locus? (ITS1 or ITS2)", "inputMode": "REQUIRED", "notes": {"Optional": "", "Dropdown": ["--locus ITS1", "--locus ITS1"]}},
                {"name": "paired_end", "arg": "--paired_end", "description": f"Paired-end", "inputMode": "INCLUDE_ON_DEMAND"},
                {"name": "max_expected_error", "arg": "--max_expected_error 3", "description": f"Maximum number of expected errors per read (> 0)", "inputMode": "REQUIRED"},
                {"name": "tax_confidence", "arg": "--tax_confidence 50", "description": "The minimum bootstrap confidence for assigning a taxonomic level", "inputMode": "REQUIRED"},
                {"name": "clustering_thresholds", "arg": "--clustering_thresholds 100,97", "description": f"Sequence similarity thresholds for OTU clustering (comma separated)", "inputMode": "REQUIRED"},
                {"name": "skip_lulu", "arg": "--skip_lulu", "description": f"Skip LULU step?", "inputMode": "INCLUDE_ON_DEMAND"},
                {"name": "alpha_diversity", "arg": "--alpha_diversity nseqs-sobs-chao-shannon-shannoneven", "description": f"Alpha diversity metrics to compute (available in mothur)", "inputMode": "REQUIRED"},
                {"name": "beta_diversity", "arg": "--beta_diversity braycurtis-thetayc-sharedsobs-sharedchao", "description": f"Beta diversity metrics to compute (available in mothur)", "inputMode": "REQUIRED"}
            ]

        },
        "memoryMB": 16000,
        "nodeCount": 1,
        "coresPerNode": 1,
        "maxMinutes": 120
    },
}

In [ ]:
# client.apps.createAppVersion(**app_def_hpc)

client.apps.patchApp(**app_def_hpc, appId=app_id_hpc, appVersion='1.0')

### Demux App Def

In [ ]:
app_id_hpc = "demux-uhhpc"
# output_dir = "${JobOwner}/jobs/${JobUUID}"
output_dir = "${JobWorkingDir}/jobs/${JobUUID}"

app_def_hpc= {
    "id": app_id_hpc,
    "version": "1.0",
    "description": "Paired-end reads demultiplexer.",
    "runtime": "ZIP",
    "runtimeOptions": ["NONE"],
    "strictFileInputs": False,
    "containerImage": "/mnt/lustre/koa/lab/cmaiki_group/cmaiki_v2_apps/demux-app-v1.0.tar.gz",
    "jobType": "BATCH",
    "jobAttributes": {
        "archiveOnAppError": True,
        "archiveSystemId": system_id_hpc,
        "archiveSystemDir": output_dir,
        "execSystemId": system_id_hpc,
        "execSystemOutputDir": output_dir,
        "parameterSet": {
            "archiveFilter": {
                "includeLaunchFiles": True
            },
            "appArgs": [
                {"name": "outdir", "arg": f"--outdir demultiplexed", "description": f"Output path", "inputMode": "REQUIRED", "notes": {"Hidden": "true"}},
                {"name": "max_mismatches", "arg": "--max_mismatches 3", "description": f"Number of allowed mismatched with (combined-paired-end) barcode(s) sequence", "inputMode": "REQUIRED"},
                {"name": "n_per_file", "arg": "--n_per_file 100000", "description": f"Number of reads per file (demultiplexing is done in parallel on each sub-file)", "inputMode": "REQUIRED"},
                {"name": "n_bases", "arg": "--n_bases 100000", "description": "Number of bases to build the error model", "inputMode": "REQUIRED"},
                {"name": "matching", "arg": "--matching auto", "description": "Order to match index with barcodes. Choices: direct, reversed, auto", "inputMode": "REQUIRED"},
                {"name": "reverseComplement", "arg": "--reverseComplement", "description": "Reverse complement I2 (or I1 if reads are single-barcoded)", "inputMode": "INCLUDE_ON_DEMAND"},
                {"name": "singleBarcoded", "arg": "--singleBarcoded", "description": "Single barcoded reads", "inputMode": "INCLUDE_ON_DEMAND"}
            ]
        },
        "memoryMB": 16000,
        "nodeCount": 1,
        "coresPerNode": 1,
        "maxMinutes": 30
    },
}

In [ ]:
# client.apps.createAppVersion(**app_def_hpc)

client.apps.patchApp(**app_def_hpc, appId=app_id_hpc, appVersion='1.0')

In [ ]:
# # List all applications available to you
# print("****************************************************")
# print("List all applications")
# print("****************************************************")
# client.apps.getApps()

### Ampliseq App Def

In [ ]:
app_id_hpc = "ampliseq-pipeline-test"
output_dir = "${JobOwner}/jobs/${JobUUID}"

app_def_hpc= {
    "id": app_id_hpc,
    "version": "0.1",
    "description": "Ampliseq pipeline with access to many of, but not all parameters. For Advanced Users wanting more ampliseq options.",
    "runtime": "ZIP",
    "runtimeOptions": ["NONE"],
    "strictFileInputs": False,
    "containerImage": "/mnt/lustre/koa/lab/cmaiki_group/cmaiki_v2_apps/ampliseq-test-pipeline-app-v0.1.tar.gz",
    "jobType": "BATCH",
    "jobAttributes": {
        "archiveOnAppError": True,
        "archiveSystemId": system_id_hpc,
        "archiveSystemDir": output_dir,
        "execSystemId": system_id_hpc,
        "execSystemOutputDir": output_dir,
        "parameterSet": {
            "archiveFilter": {
                "includeLaunchFiles": True
            },
            "appArgs": AMPLISEQ_TEST_ARGS
        },
        "memoryMB": 16000,
        "nodeCount": 1,
        "coresPerNode": 1,
        "maxMinutes": 360
    },
}

In [ ]:
# client.apps.createAppVersion(**app_def_hpc)

client.apps.patchApp(**app_def_hpc, appId=app_id_hpc, appVersion='0.1')

### Ampliseq ITS App Def

In [ ]:
app_id_hpc = "ampliseq-ITS-pipeline-uhhpc"
output_dir = "${JobOwner}/jobs/${JobUUID}"

app_def_hpc= {
    "id": app_id_hpc,
    "version": "0.1",
    "description": "Ampliseq ITS pipeline with ITS specific parameter set using test files.",
    "runtime": "ZIP",
    "runtimeOptions": ["NONE"],
    "strictFileInputs": False,
    "containerImage": "/mnt/lustre/koa/lab/cmaiki_group/cmaiki_v2_apps/ampliseq-ITS-pipeline-app-v0.1.tar.gz",
    "jobType": "BATCH",
    "jobAttributes": {
        "archiveOnAppError": True,
        "archiveSystemId": system_id_hpc,
        "archiveSystemDir": output_dir,
        "execSystemId": system_id_hpc,
        "execSystemOutputDir": output_dir,
        "parameterSet": {
            "archiveFilter": {
                "includeLaunchFiles": True
            },
            "appArgs": AMPLISEQ_ITS_AMP_ARGS

        },
        "memoryMB": 32000,
        "nodeCount": 1,
        "coresPerNode": 1,
        "maxMinutes": 360
    },
}

In [ ]:
# client.apps.createAppVersion(**app_def_hpc)

client.apps.patchApp(**app_def_hpc, appId=app_id_hpc, appVersion='0.1')

### Ampliseq 16S App Def

In [ ]:
app_id_hpc = "ampliseq-16S-pipeline-uhhpc"
output_dir = "${JobOwner}/jobs/${JobUUID}"

app_def_hpc= {
    "id": app_id_hpc,
    "version": "0.2",
    "description": "Ampliseq 16S pipeline with 16S specific parameter set using test files.",
    "runtime": "ZIP",
    "runtimeOptions": ["NONE"],
    "strictFileInputs": False,
    "containerImage": "/mnt/lustre/koa/lab/cmaiki_group/cmaiki_v2_apps/ampliseq-16S-pipeline-app-v0.2.tar.gz",
    "jobType": "BATCH",
    "jobAttributes": {
        "archiveOnAppError": True,
        "archiveSystemId": system_id_hpc,
        "archiveSystemDir": output_dir,
        "execSystemId": system_id_hpc,
        "execSystemOutputDir": output_dir,
        "parameterSet": {
            "archiveFilter": {
                "includeLaunchFiles": True
            },
            "appArgs": AMPLISEQ_16S_AMP_ARGS

        },
        "memoryMB": 32000,
        "nodeCount": 1,
        "coresPerNode": 1,
        "maxMinutes": 360
    },
}

In [ ]:
# client.apps.createAppVersion(**app_def_hpc)

client.apps.patchApp(**app_def_hpc, appId=app_id_hpc, appVersion='0.2')

### Delete an app

In [ ]:
# client.apps.deleteApp(appId="ampliseq-condensed-pipeline-test")

In [ ]:
# # Get details for the application you created
# print("****************************************************")
# print("Fetch application: ")
# print("****************************************************")
# client.apps.getAppLatestVersion(appId=app_id_hpc)

### Demux  Job Def

In [ ]:
# # Submit job to run the application
# job_def_hpc = {
#     "name": "demux-test-files",
#     "description":"Running the demux app with test files",
#     "appId": app_id_hpc,
#     "appVersion": '0.7',
#     "execSystemId":system_id_hpc,
#     "jobType": "BATCH",
# }

# # Submit a job
# job_response_hpc=client.jobs.submitJob(**job_def_hpc)

### Get Job submission response


In [ ]:
# # Get Job submission response
# print(job_response_hpc)

### Get Job UUID from the submission response


In [ ]:
# Get job uuid from the job submission response
job_uuid_hpc=job_response_hpc.uuid

print("****************************************************")
print("Job UUID: " + job_uuid_hpc)
print("****************************************************")

### Check the status of the job


In [ ]:
# Check the status of the job

job_uuid_hpc = ""
print("****************************************************")
print(client.jobs.getJobStatus(jobUuid=job_uuid_hpc))
print("****************************************************")

In [ ]:
print(client.jobs.getJobHistory(jobUuid="5fb03e83-3888-4139-92c2-6fd334b3916f-007"))

### Download output of the job


In [ ]:
# # Once the job is in the FINISHED state, you can download output of the job
# jobs_output_hpc= client.jobs.getJobOutputDownload(jobUuid=job_uuid_hpc,outputPath='tapisjob.out')

# print("Job Output file:")
# print("****************************************************")
# print(jobs_output_hpc)
# print("****************************************************")

### Cancel a job


In [ ]:
# If necessary, you can cancel a long running job.
job_uuid_hpc = ""
client.jobs.cancelJob(jobUuid=job_uuid_hpc)

In [ ]:
print("****************************************************")
print(client.jobs.getJobStatus(jobUuid=job_uuid_hpc))
print("****************************************************")

## Share System and App

In [ ]:
system_id_hpc

In [ ]:
# Making your execution system Public
client.systems.shareSystemPublic(systemId=system_id_hpc)

In [ ]:
# Share execution system with specific user
client.systems.shareSystem(systemId=system_id_hpc, users=["mcleana"])

In [ ]:
# Get Share info on the system
client.systems.getShareInfo(systemId=system_id_hpc)

In [ ]:
# Making the app public
client.apps.shareAppPublic(appId=app_id_hpc)

In [ ]:
# Share apps with a specific user
apps_to_share = ['16S-v0.0.2-pipeline-uhhpc', 'demux-uhhpc', 'ITS-pipeline-uhhpc']
users = ['mcleana']

for app_id in apps_to_share:
    try:
        client.apps.shareApp(appId=app_id, users=users)
        print(f"Successfully shared {app_id}")
    except Exception as e:
        print(f"Failed to share {app_id}: {e}")

In [ ]:
# Get Share info on the app
client.apps.getShareInfo(appId=app_id_hpc)
# Now any user in the tenant should be able to run your application

In [ ]:
# Unsharing public app
#client.apps.unShareAppPublic(appId=app_id_hpc)

In [ ]:
## You should now be able to run any public apps
'''
pa= {
    "parameterSet": {
    "appArgs": [
        {"arg": "--text 'I am happy today'"}

        ]
    }}

# Submit a job
job_response_hpc_email=client.jobs.submitJob(name='sentiment analysis',description='sentiment analysis with hugging face transformer pipelines',appId=app_id_hpc,appVersion='0.1',execSystemId=system_id_hpc,subscriptions= [ { "description": "Test subscriptions", "eventCategoryFilter": "ALL","deliveryTargets": [ { "deliveryMethod": "EMAIL","deliveryAddress":"<Enter your email>"}] }],**pa)
'''